# Epsilon Fund - Walk-Forward Validation
Uses `infrastructure/walk_forward/` to run rolling Optuna optimisation and evaluate OOS robustness. Ideal to use after finding strategy that seems to work using backtesting framework to ensure logic is valid.

---
### Iteration Guidelines

**Overfitting the iteration process:** Each time you inspect OOS results and adjust parameters, you leak OOS information into your design decisions. Cap yourself at **3–4 iterations** — first run with everything free, second with obvious fixes from CV + plateau analysis, third to tighten remaining params. 

If the strategy still shows heavy overfitting signals after three passes, **the problem is the strategy architecture, not the parameters**.

**WFE:** Walk-forward efficiency - examine IS/OOS ratio (simplest).

**Pertubation degradation:** Examine pertubation table to see if degradation reduces across runs.


| Signal | Meaning | Action |
|--------|---------|--------|
| IS Sharpe drops, OOS Sharpe holds or rises, WFE improves | Removing noise-fitting degrees of freedom | Continue iterating |
| Perturbation degradation shrinks across iterations | Parameters becoming more robust | Continue iterating |
| N/A plateau params decreasing across iterations | Strategy becoming more tolerant of parameter movement | Continue iterating |
| WFE improvement flattens (e.g. 0.55 → 0.65 → 0.67) | Diminishing returns — further fixes won't help much | Stop iterating |
| IS and OOS both decline but WFE rises (IS falls faster) | Constraining away real signal, not just noise | Stop iterating |
| OOS Sharpe keeps declining despite "better" param setup | Overfitting the iteration process itself | Stop — problem is strategy architecture, not parameters |
| WFE decreases after fixing a parameter | Locked in a param that was legitimately adapting across folds | Unfix that parameter and re-run |

---

In [1]:
import sys
import os
import pandas as pd
import numpy as np


# ── repo root — works on both Mac and Windows ────────────────────────────────
ROOT = os.path.expanduser('~/Desktop/epsilon/github/Epsilon-Quant-Research')
# ROOT = r'C:\Users\user\Documents\Epsilon Fund\Epsilon-Quant-Research'  # ← Windows path
# ─────────────────────────────────────────────────────────────────────────────

sys.path.append(os.path.join(ROOT, 'infrastructure', 'data'))
sys.path.append(os.path.join(ROOT, 'infrastructure', 'walkforward'))
sys.path.append(os.path.join(ROOT, 'infrastructure', 'backtester'))


from binance_client import get_binance_client, get_data
from wf_engine import walk_forward, plateau_analysis, plateau_summary, perturbation_test, cost_stress_test
from wf_visualizer import plot_walk_forward_results, plot_plateau_analysis
from engine import backtest

---
## Data

**Pairs** — any Binance pair in `BASEQUOTE` format (e.g. `BTCUSDT`, `ETHUSDT`, `SOLUSDT`, `BNBUSDT`).  
Verify availability at [binance.com/en/trade](https://www.binance.com/en/trade).

**Intervals** — `'1m'` `'5m'` `'15m'` `'1h'` `'4h'` `'1d'` `'1w'`

**Lookback** — days of history: must be >= (train_bars + test_bars) * n_folds desired

In [2]:
SYMBOL   = 'LINKUSDT'
INTERVAL = '1d'
LOOKBACK = 2150 


# ── Multiple pairs (optional) ──────────────────────────────────────────────────
# SYMBOLS = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT']
# data_dict = get_multiple_data(client, SYMBOLS, INTERVAL, LOOKBACK)
# Access via: data_dict['BTCUSDT_1d'], data_dict['ETHUSDT_1d'] ...
# ──────────────────────────────────────────────────────────────────────────────

client   = get_binance_client()
df = get_data(client, SYMBOL, INTERVAL, LOOKBACK)
print(f'Data: {df.index[0].date()} → {df.index[-1].date()}  ({len(df)} bars)')
df.tail(3)

Data: 2020-05-21 → 2026-04-09  (2150 bars)


,Open,High,Low,Close,Volume
Time,,,,,
2026-04-07,8.80,9.42,8.57,9.29,5126596.75
2026-04-08,9.29,9.34,8.84,8.85,3690230.33
2026-04-09,8.85,8.86,8.68,8.80,2180928.35


---
## Parameter Configuration

Define which parameters to optimise and anchor - **recommended to do after strategy writeup**

`FIXED_PARAMS`: choose parameters with CV < 0.15 from prior stability run, cross referencing with pertubation verdict table to reduce search space, improve OOS credibility.


In [18]:
# ── parameter search space ────────────────────────────────────────────────────
# Format: 'param_name': ('int' | 'float', lo, hi)
# Only keys present in PARAMS above are searched — remove a key from PARAMS to exclude it entirely.

PARAM_DEFS = {
    'ema_span':          ('int',   5,    40),
    'swing_caution':     ('int',   3,    14),
    'swing_stop':        ('int',   3,    10),
    'atr_caution':       ('int',   10,   30),
    'atr_stop':          ('int',   20,   30),
    'atr_size':          ('int',   3,    14),
    'adx_override':      ('int',   40,   80),
    'stop_atr_scale':    ('float', 0.5,  2.0),
    'risk_per_trade':    ('float', 0.005, 0.05),
    'max_leverage':      ('float', 1.0,  3.0),
    'stop_mult_pos_caution': ('float', 0.1, 0.9),
    'stop_mult_pos_normal':  ('float', 0.8, 2.0),
    'stop_mult_ent_both':    ('float', 0.8, 2.5),
    'stop_mult_ent_caution': ('float', 0.1, 0.9),
    'stop_mult_ent_short':   ('float', 0.1, 1.5),
    'stop_mult_ent_normal':  ('float', 0.5, 1.5),
    'vol_ma_period':  ('int',   10,  40),   # rolling window for volume MA
    'obv_ma_period':  ('int',   10,  40),   # OBV smoothing window
    'obv_lookback':   ('int',   10,  30),   # bars back to compare price vs OBV direction
}

# ── anchored params (set after a stability run; empty first time) ─────────────
# These bypass Optuna and are held constant across all folds.
# Populate using stability_df results: fix params with CV < 0.15
FIXED_PARAMS = { 
    'risk_per_trade': 0.046,
    'max_leverage': 2.5,

    'stop_mult_pos_normal': 1,
    'stop_mult_ent_normal': 1,

    'obv_ma_period': 34,

    }

### *Guide to parameter anchoring*

|  | Robust Plateau| Fragile Plateau |
|--|-------------------|-------------------|
| Low CV | Stable across folds and insensitive to exact value - keep free| Looks stable but is fitting the same noise patterns - fix at concensus|
| High CV | Parameter unimportant - fix at any reasonable value | Unstable across folds and sitting on a cliff - strong candidate to eliminate |

Copy-paste plateau analysis table above into fixed params section and decide manually on which to fix/keep free.c

---
## Strategy

**CRUCIAL** - Strategy logic needs to work well in backtesting notebook before running here, saves time not running walk-forward for a broken strategy.

**Available columns:** `Open` `High` `Low` `Close` `Volume`

**Required output:** a `position` column — `1` long · `0` flat · `-1` short  
**Optional:** `position_size` column (0–1) to use fractional capital

> The engine applies a 1-bar execution lag automatically. Inside the strategy loop, use `prev` for signal decisions and `curr` for execution — no manual shifting needed.

**To implement your strategy:**
1. Write strategy logic — compute indicators, signals, and execution loop: use `param_name`for those to be searched
2. Update `indicator_cols` to list your longest-warmup indicators — the engine uses this to clean NaN rows after OOS trimming


In [19]:
def my_strategy(df_slice: pd.DataFrame, params: dict) -> pd.DataFrame:

    df = df_slice.copy()

    # ── strategy logic ────────────────────────────────────────────────────────
    df['EMA']          = df['Close'].ewm(span=params['ema_span'], adjust=False).mean()
    df['Swing_Hi_Cau'] = df['High'].rolling(params['swing_caution']).max()
    df['Swing_Lo_Cau'] = df['Low'].rolling(params['swing_caution']).min()
    df['Swing_Hi_Stp'] = df['High'].rolling(params['swing_stop']).max()

    def atr(period):
        hl = df['High'] - df['Low']
        hc = (df['High'] - df['Close'].shift(1)).abs()
        lc = (df['Low']  - df['Close'].shift(1)).abs()
        return pd.concat([hl, hc, lc], axis=1).max(axis=1).ewm(span=period, adjust=False).mean()

    df['ATR_Cau'] = atr(params['atr_caution'])
    df['ATR_Stp'] = atr(params['atr_stop'])
    df['ATR_Sz']  = atr(params['atr_size'])

    up    = df['High'].diff();  down = -df['Low'].diff()
    pdm   = up.where((up > down) & (up > 0), 0.0)
    ndm   = down.where((down > up) & (down > 0), 0.0)
    atr14 = atr(14)
    pdi   = 100 * pdm.ewm(span=14, adjust=False).mean() / atr14
    ndi   = 100 * ndm.ewm(span=14, adjust=False).mean() / atr14
    dx    = (100 * (pdi - ndi).abs() / (pdi + ndi).replace(0, np.nan)).fillna(0)
    df['ADX_14'] = dx.ewm(span=14, adjust=False).mean()

    df['Vol_MA']  = df['Volume'].rolling(params['vol_ma_period']).mean()
    direction     = df['Close'].diff().apply(lambda x: 1 if x > 0 else -1)
    df['OBV']     = (df['Volume'] * direction).cumsum()
    df['OBV_MA']  = df['OBV'].rolling(params['obv_ma_period']).mean()


    df['Caution_OBV']   = (df['Close'] > df['Close'].shift(params['obv_lookback'])) & (df['OBV'] < df['OBV_MA'])
    df['Caution_Long']  = ((df['Swing_Hi_Cau'] - df['Low']) > 1.5 * df['ATR_Cau']) | df['Caution_OBV']
    df['Caution_Short'] = ((df['High'] - df['Swing_Lo_Cau']) > 1.5 * df['ATR_Cau']) | (df['Close'] > df['EMA'])
    df['Entry_Long'] = (df['Close'] > df['EMA']) & (~df['Caution_Long'] | (df['ADX_14'] > params['adx_override'])) & (df['Volume'] > df['Vol_MA'])
    df['position_size_raw'] = (params['risk_per_trade'] / (df['ATR_Sz'] / df['Close'])).clip(0.1, params['max_leverage'])

    n             = len(df)
    position      = [0]     * n
    position_size = [0.0]   * n
    stop_arr      = [np.nan] * n
    in_position   = 0
    stop_loss     = np.nan
    current_size  = 0.0

    for i in range(1, n):
        prev = df.iloc[i - 1]
        curr = df.iloc[i]

        if in_position == 0:
            if prev['Entry_Long']:
                in_position  = 1
                current_size = curr['position_size_raw']
                cl = prev['Caution_Long']; cs = prev['Caution_Short']
                if cl and cs: sm = params['stop_mult_ent_both']
                elif cl:        sm = params['stop_mult_ent_caution']
                else:           sm = params['stop_mult_ent_normal']
                stop_loss = curr['Swing_Hi_Stp'] - curr['ATR_Stp'] * sm * params['stop_atr_scale']
        else:
            if prev['Close'] < stop_loss:
                in_position  = 0
                current_size = 0.0
                stop_loss    = np.nan
            else:
                sm        = params['stop_mult_pos_caution'] if curr['Caution_Long'] else params['stop_mult_pos_normal']
                stop_loss = max(stop_loss, curr['Swing_Hi_Stp'] - curr['ATR_Stp'] * sm * params['stop_atr_scale'])

        position[i]      = in_position
        position_size[i] = current_size
        stop_arr[i]      = stop_loss

    df['position']      = position
    df['position_size'] = position_size
    df['stop_loss']     = stop_arr

    # ── cleanup ───────────────────────────────────────────────────────────────
    indicator_cols = ['EMA', 'ATR_Cau', 'ADX_14', 'Swing_Hi_Cau', 'Vol_MA', 'OBV_MA']
    df['position']      = df['position'].fillna(0).astype(int)
    df['position_size'] = df['position_size'].fillna(0.0)
    df['stop_loss']     = df['stop_loss'].fillna(0.0)

    return df, indicator_cols

---
## Run Walk-Forward
Simulates how the strategy would have performed if re-optimised periodically
in live trading, and exposes whether good IS performance survives unseen data.

**Folds Setup**
| Parameter | Description | Guidance |
|---|---|---|
| `TRAIN_BARS` | Bars per training window | Aim for 2:1 to 3:1 ratio vs `TEST_BARS` |
| `TEST_BARS` | Bars per test window | `365` = ~1 year on daily data |
| `BURNIN_BARS` | Bars prepended to test for indicator warmup | Match your longest indicator period |
| `N_TRIALS` | Optuna trials per fold | 300–500 for daily; more = better but slower.10-20 trials per free parameter to avoid overfit |
| `COST` | Round-trip cost per trade | `0.001` = 0.1% |

Use `N_TRIALS` as robustness dia: if OOS degrades sharply as you increase it from 100→200→300, direct signal your parameter space still has too many degrees of freedom relative to the information content of the training window (consider decreasing). 

**Score and Rejection** — use to calibrate what Optuna optimises IS: default `score_fn(m)` uses weighted basket of Sharpe, Calmar and Return, normalised using their "max" value; default `reject_fn(m)` discards runs failing certain criteria that limits credibility.

> Pay attention to the **degradation ratio** — OOS/IS Sharpe reveals overfitting.

In [20]:
# ── walk-forward windows ──────────────────────────────────────────────────────
TRAIN_BARS  = 1050  
TEST_BARS   = 137   
BURNIN_BARS = 100   
N_TRIALS    = 500  
COST        = 0.001 

# ── SCORING FUNCTION ──────────────────────────────────────────────────────────
# Modify weights or swap components. Must return a float (higher = better).

def score_fn(m):
    SHARPE_MAX = 2
    CALMAR_MAX = 3
    RETURN_MAX = 4 #Calibrate to max of IS periods

    calmar = m['total_return'] / abs(m['max_drawdown']) if m['max_drawdown'] != 0 else 0

    s = np.clip(m['sharpe_ratio']  / SHARPE_MAX, 0, 1)
    c = np.clip(calmar             / CALMAR_MAX, 0, 1)
    r = np.clip(m['total_return']  / RETURN_MAX, 0, 1)

    return 0.50 * s + 0.30 * c + 0.20 * r

# ── REJECTION CRITERIA ────────────────────────────────────────────────────────
# Trials that return True are discarded (score → -999).

def reject_fn(m):
    if m is None:                      return True
    if m['num_trades']    < 10:        return True
    if m['win_rate']      < 0.3:      return True
    if m['max_drawdown']  < -0.7:     return True
    if m['profit_factor'] < 0.6:       return True
    return False


results = walk_forward(
    df           = df,
    strategy_fn  = my_strategy,
    param_defs   = PARAM_DEFS,
    fixed_params = FIXED_PARAMS,
    train_bars   = TRAIN_BARS,
    test_bars    = TEST_BARS,
    burnin_bars  = BURNIN_BARS,
    n_trials     = N_TRIALS,
    cost         = COST,
    score_fn     = score_fn,    # ← your notebook definition
    reject_fn    = reject_fn,   # ← your notebook definition
    save_csv     = None,
)

UPDATED WALK_FORWARD FILE IS RUNNING
Walk-forward: 8 fold(s)  train=1050  test=137  burnin=100  trials=500
  Fold 1: train 2020-05-21 → 2023-04-05  | test 2023-04-06 → 2023-08-20
  Fold 2: train 2020-10-05 → 2023-08-20  | test 2023-08-21 → 2024-01-04
  Fold 3: train 2021-02-19 → 2024-01-04  | test 2024-01-05 → 2024-05-20
  Fold 4: train 2021-07-06 → 2024-05-20  | test 2024-05-21 → 2024-10-04
  Fold 5: train 2021-11-20 → 2024-10-04  | test 2024-10-05 → 2025-02-18
  Fold 6: train 2022-04-06 → 2025-02-18  | test 2025-02-19 → 2025-07-05
  Fold 7: train 2022-08-21 → 2025-07-05  | test 2025-07-06 → 2025-11-19
  Fold 8: train 2023-01-05 → 2025-11-19  | test 2025-11-20 → 2026-04-05

Fixed (5): ['risk_per_trade', 'max_leverage', 'stop_mult_pos_normal', 'stop_mult_ent_normal', 'obv_ma_period']
Free  (14): ['ema_span', 'swing_caution', 'swing_stop', 'atr_caution', 'atr_stop', 'atr_size', 'adx_override', 'stop_atr_scale', 'stop_mult_pos_caution', 'stop_mult_ent_both', 'stop_mult_ent_caution', 'sto

  0%|          | 0/500 [00:00<?, ?it/s]


  IS  → Sharpe: 1.75  Return: 302.47%  DD: -17.18%  Calmar: 3.62  Trades: 71
  OOS → Sharpe: 0.24  Return: 0.83%  DD: -23.40%  Calmar: 0.10  Trades: 9

  Best params: {'risk_per_trade': 0.046, 'max_leverage': 2.5, 'stop_mult_pos_normal': 1, 'stop_mult_ent_normal': 1, 'obv_ma_period': 34, 'ema_span': 27, 'swing_caution': 5, 'swing_stop': 3, 'atr_caution': 19, 'atr_stop': 26, 'atr_size': 14, 'adx_override': 45, 'stop_atr_scale': 0.7869859846386368, 'stop_mult_pos_caution': 0.7798246471219584, 'stop_mult_ent_both': 2.0274585333445483, 'stop_mult_ent_caution': 0.6769255109831603, 'stop_mult_ent_short': 0.9973856028340503, 'vol_ma_period': 10, 'obv_lookback': 14}

────────────────────────────────────────────────────────────
Fold 2/8  train: 2020-10-05 → 2023-08-20  test: 2023-08-21 → 2024-01-04


  0%|          | 0/500 [00:00<?, ?it/s]


  IS  → Sharpe: 0.98  Return: 79.12%  DD: -19.28%  Calmar: 1.16  Trades: 62
  OOS → Sharpe: 0.90  Return: 7.63%  DD: -9.45%  Calmar: 2.29  Trades: 9

  Best params: {'risk_per_trade': 0.046, 'max_leverage': 2.5, 'stop_mult_pos_normal': 1, 'stop_mult_ent_normal': 1, 'obv_ma_period': 34, 'ema_span': 6, 'swing_caution': 11, 'swing_stop': 3, 'atr_caution': 22, 'atr_stop': 30, 'atr_size': 6, 'adx_override': 70, 'stop_atr_scale': 0.7403328081134036, 'stop_mult_pos_caution': 0.18394736982567156, 'stop_mult_ent_both': 0.9100265803760781, 'stop_mult_ent_caution': 0.8566314761224586, 'stop_mult_ent_short': 0.88348449639441, 'vol_ma_period': 40, 'obv_lookback': 18}

────────────────────────────────────────────────────────────
Fold 3/8  train: 2021-02-19 → 2024-01-04  test: 2024-01-05 → 2024-05-20


...


  IS  → Sharpe: 1.38  Return: 184.82%  DD: -22.20%  Calmar: 1.98  Trades: 76
  OOS → Sharpe: -2.95  Return: -15.13%  DD: -15.42%  Calmar: -2.30  Trades: 7

  Best params: {'risk_per_trade': 0.046, 'max_leverage': 2.5, 'stop_mult_pos_normal': 1, 'stop_mult_ent_normal': 1, 'obv_ma_period': 34, 'ema_span': 12, 'swing_caution': 3, 'swing_stop': 6, 'atr_caution': 25, 'atr_stop': 25, 'atr_size': 9, 'adx_override': 79, 'stop_atr_scale': 0.9253580363792236, 'stop_mult_pos_caution': 0.533783719565318, 'stop_mult_ent_both': 0.9254785858715524, 'stop_mult_ent_caution': 0.17759752692639255, 'stop_mult_ent_short': 1.3263402266035327, 'vol_ma_period': 27, 'obv_lookback': 21}

────────────────────────────────────────────────────────────
Fold 7/8  train: 2022-08-21 → 2025-07-05  test: 2025-07-06 → 2025-11-19


  0%|          | 0/500 [00:00<?, ?it/s]


  IS  → Sharpe: 1.36  Return: 154.91%  DD: -14.88%  Calmar: 2.58  Trades: 71
  OOS → Sharpe: 0.96  Return: 8.01%  DD: -8.28%  Calmar: 2.75  Trades: 10

  Best params: {'risk_per_trade': 0.046, 'max_leverage': 2.5, 'stop_mult_pos_normal': 1, 'stop_mult_ent_normal': 1, 'obv_ma_period': 34, 'ema_span': 39, 'swing_caution': 4, 'swing_stop': 9, 'atr_caution': 23, 'atr_stop': 29, 'atr_size': 13, 'adx_override': 77, 'stop_atr_scale': 0.8190513884354488, 'stop_mult_pos_caution': 0.12073938232152705, 'stop_mult_ent_both': 1.8251549073134905, 'stop_mult_ent_caution': 0.817286001597883, 'stop_mult_ent_short': 0.5041094119262343, 'vol_ma_period': 32, 'obv_lookback': 21}

────────────────────────────────────────────────────────────
Fold 8/8  train: 2023-01-05 → 2025-11-19  test: 2025-11-20 → 2026-04-05


  0%|          | 0/500 [00:00<?, ?it/s]


  IS  → Sharpe: 1.40  Return: 148.79%  DD: -17.70%  Calmar: 2.11  Trades: 96
  OOS → Sharpe: -3.73  Return: -20.37%  DD: -20.37%  Calmar: -2.23  Trades: 4

  Best params: {'risk_per_trade': 0.046, 'max_leverage': 2.5, 'stop_mult_pos_normal': 1, 'stop_mult_ent_normal': 1, 'obv_ma_period': 34, 'ema_span': 29, 'swing_caution': 4, 'swing_stop': 7, 'atr_caution': 16, 'atr_stop': 28, 'atr_size': 13, 'adx_override': 45, 'stop_atr_scale': 0.5001861271197094, 'stop_mult_pos_caution': 0.4110890822351335, 'stop_mult_ent_both': 1.5112452136148722, 'stop_mult_ent_caution': 0.5495240091440945, 'stop_mult_ent_short': 0.39576282958086983, 'vol_ma_period': 40, 'obv_lookback': 14}

════════════════════════════════════════════════════════════
WALK-FORWARD SUMMARY
════════════════════════════════════════════════════════════

Out-of-sample across 8 fold(s):
  Avg Sharpe:       -0.32
  Avg Return:       -0.5%
  Avg Max Drawdown: -14.9%
  Avg Calmar:       1.40
  Avg Trades/fold:  8
  Folds profitable: 5/8


---
## Granular Results and Parameter Stability

Per-fold IS vs OOS performance. Each row is one fold — compare `train_*` vs `test_*` columns to assess overfitting.

| Column | Description |
|---|---|
| `*_sharpe` `*_return` `*_drawdown` `*_calmar` | Core performance metrics |
| `*_trades` `*_winrate` `*_profit_factor` | Trade statistics |
| `optuna_score` | Best score achieved on training window |
| `param_*` | Best parameter values per fold e.g. `param_ema_span` |

**Concensus Parameters** - use to anchor: the engine determines stability using the coefficient of variation (CV) — the standard deviation of a parameter's best values across all folds divided by their median.

>CV < 0.15: indicates the strategy  relies on value rather than it being noise-fitted to a specific period — making it safe to fix for future runs. A high CV means the parameter is period-sensitive and should stay free.

In [21]:
# ── fold summary table ────────────────────────────────────────────────────────
display_cols = [
    'train_start', 'train_end',
    'test_start', 'test_end',
    'train_sharpe', 'test_sharpe',
    'train_return', 'test_return',
    'train_drawdown', 'test_drawdown',
    'test_trades',  'test_winrate', 'optuna_score',
]
display(results['results_df'][display_cols].round(2))

# ── parameter stability ───────────────────────────────────────────────────────
stability = results['stability_df'].copy()
stability['stable'] = stability['stable'].map({True: '✓', False: ''})
stability['fixed']  = stability['fixed'].map({True: '✓', False: ''})
stability = stability[['param', 'median', 'std', 'cv', 'stable', 'fixed']].round(2)
display(stability.sort_values('cv'))

# ── consensus params ──────────────────────────────────────────────────────────
stable = results['stability_df'][results['stability_df']['cv'] < 0.15]

print('Stable parameters (CV < 0.15) — copy into FIXED_PARAMS:')
for _, row in stable.iterrows():
    v = results['consensus_params'][row['param']]
    v_fmt = int(round(v)) if isinstance(v, float) and v == int(v) else round(v, 4) if isinstance(v, float) else v
    print(f"    '{row['param']}': {v_fmt},")
    
print('\nConsensus parameters (median across folds):')
for k, v in results['consensus_params'].items():
    print(f'  {k:<30} = {round(v, 2) if isinstance(v, float) else v}')

,train_start,train_end,test_start,test_end,train_sharpe,test_sharpe,train_return,test_return,train_drawdown,test_drawdown,test_trades,test_winrate,optuna_score
0,2020-05-21,2023-04-05,2023-04-06,2023-08-20,1.75,0.24,3.02,0.01,-0.17,-0.23,9,0.67,0.89
1,2020-10-05,2023-08-20,2023-08-21,2024-01-04,0.98,0.90,0.79,0.08,-0.19,-0.09,9,0.56,0.59
2,2021-02-19,2024-01-04,2024-01-05,2024-05-20,1.19,2.00,1.11,0.18,-0.28,-0.08,7,0.57,0.65
3,2021-07-06,2024-05-20,2024-05-21,2024-10-04,1.15,-1.38,2.22,-0.19,-0.27,-0.25,6,0.33,0.70
4,2021-11-20,2024-10-04,2024-10-05,2025-02-18,1.09,1.37,1.03,0.16,-0.18,-0.09,14,0.43,0.62
5,2022-04-06,2025-02-18,2025-02-19,2025-07-05,1.38,-2.95,1.85,-0.15,-0.22,-0.15,7,0.14,0.74
6,2022-08-21,2025-07-05,2025-07-06,2025-11-19,1.36,0.96,1.55,0.08,-0.15,-0.08,10,0.50,0.72
7,2023-01-05,2025-11-19,2025-11-20,2026-04-05,1.40,-3.73,1.49,-0.20,-0.18,-0.20,4,0.00,0.73


,param,median,std,cv,stable,fixed
9,max_leverage,2.50,0.00,0.00,✓,✓
15,stop_mult_ent_normal,1.00,0.00,0.00,✓,✓
11,stop_mult_pos_normal,1.00,0.00,0.00,✓,✓
17,obv_ma_period,34.00,0.00,0.00,✓,✓
8,risk_per_trade,0.05,0.00,0.00,✓,✓
4,atr_stop,29.00,1.71,0.06,✓,
18,obv_lookback,17.00,3.28,0.19,,
6,adx_override,65.50,14.30,0.22,,
3,atr_caution,20.50,4.68,0.23,,
12,stop_mult_ent_both,1.55,0.43,0.27,,


Stable parameters (CV < 0.15) — copy into FIXED_PARAMS:
    'atr_stop': 29,
    'risk_per_trade': 0.046,
    'max_leverage': 2.5,
    'stop_mult_pos_normal': 1,
    'stop_mult_ent_normal': 1,
    'obv_ma_period': 34,

Consensus parameters (median across folds):
  ema_span                       = 17
  swing_caution                  = 4
  swing_stop                     = 6
  atr_caution                    = 20
  atr_stop                       = 29
  atr_size                       = 8
  adx_override                   = 66
  stop_atr_scale                 = 0.8
  risk_per_trade                 = 0.05
  max_leverage                   = 2.5
  stop_mult_pos_caution          = 0.39
  stop_mult_pos_normal           = 1.0
  stop_mult_ent_both             = 1.55
  stop_mult_ent_caution          = 0.61
  stop_mult_ent_short            = 0.94
  stop_mult_ent_normal           = 1.0
  vol_ma_period                  = 34
  obv_ma_period                  = 34
  obv_lookback                   = 17


---
## Parameter Robustness Checks

### Plateau Analysis
Sweep each free parameter across its range while holding others at consensus (median) value then evaluates the `score` at each point by backtesting over entire lookback.

The stability table (CV across folds) tells you *"does the optimizer consistently pick the same value?"*  

Plateau analysis tells you *"if that value were slightly wrong, would performance collapse?"*  

**Plateau %** - what fraction of each parameter's range stays within `threshold`% (default 20) of peak score: >60% = `robust plateau`, 30–60% = `moderate`, <30% = `fragile` (consider anchoring). `N/A` means every sweep point failed rejection filters — the strategy is completely intolerant of movement on that dimension.

>Run time: `n_free_params` × `n_steps`

### Perturbation test
Jitters all free parameters by ±5/10/20% of their range (50 random samples per offset range). Measures how much the score degrades vs the base

Test whether optimum is a broad hill in `#free params`-D space or a narrow spike

**>15%:** fragile optimum, consider reducing free parameters

In [22]:
# ── 1-D sensitivity sweeps around consensus params ─────────────────────────
sweep_results = plateau_analysis(
    df           = df,
    strategy_fn  = my_strategy,
    base_params  = results['consensus_params'],
    param_defs   = PARAM_DEFS,
    fixed_params = FIXED_PARAMS,
    cost         = COST,
    n_steps      = 20, #Adjust for number of steps around concensus per parameter
)

# ── text verdicts ──────────────────────────────────────────────────────────
verdict_df = plateau_summary(
    sweep_results,
    base_params = results['consensus_params'],
    stability_df = results['stability_df'],  
    threshold   = 0.15, #Adjust for % around peak score
)


# ── neighbouahood perturbation ────────────────────────────────────────────
# Randomly jitters ALL free params simultaneously.
# If mean score degrades >15% at ±10% offset, the optimum is fragile.

perturb_df = perturbation_test(
    df           = df,
    strategy_fn  = my_strategy,
    base_params  = results['consensus_params'],
    param_defs   = PARAM_DEFS,
    fixed_params = FIXED_PARAMS,
    cost         = COST,
    pct_offsets  = (0.05, 0.10, 0.20),   # ±5%, ±10%, ±20% of range
    n_samples    = 50,                     # random perturbations per offset level
)


═══════════════════════════════════════════════════════════════════════════════════════════
PLATEAU ANALYSIS — PARAMETER ROBUSTNESS
═══════════════════════════════════════════════════════════════════════════════════════════
Parameter                 Consensus Peak Score  Plateau %  Fold CV Verdict                 
──────────────────────── ────────── ────────── ────────── ──────── ────────────────────────
atr_size                          8      0.118      100.0%    0.576 Robust                  
stop_mult_ent_both           1.5538      0.115      100.0%    0.274 Robust                  
stop_mult_ent_caution        0.6132      0.114      100.0%    0.456 Robust                  
stop_mult_ent_short          0.9404      0.114      100.0%    0.352 Robust                  
atr_stop                         29      0.149       72.7%    0.059 Robust                  
stop_mult_pos_caution        0.3941      0.142       60.0%    0.660 Robust                  
swing_stop                       

### 1-D sweep charts:
| Element | Meaning | Good | Bad |
|---------|---------|------|-----|
| **Blue curve** | Composite score at each value of the parameter, with all others held at consensus | Flat-topped curve — performance is insensitive to the exact value | Narrow spike — optimizer latched onto one specific value, everything nearby is worse |
| **Red dashed line** | Where the consensus value sits | On the flat top of the curve | On a steep slope or at the edge of a cliff |
| **Green dashed line** | Cutoff at 80% of peak score — the boundary between plateau and non-plateau | Blue curve stays above this line across most of the range | Blue curve dips below it quickly either side of the peak |
| **Green shading** | Plateau region — all values where the score stays within 20% of the peak | Wide green band spanning most of the range (robust) | Thin sliver or no shading at all (fragile/overfit) |


 If concensus on steep slope: parameter **REGIME SENSITIVE** - do not fix, backtests are disagreeing, want to fix parameters on flat top.

In [17]:
# ── visual sweep curves ───────────────────────────────────────────────────
plot_plateau_analysis(
    sweep_results    = sweep_results,
    consensus_params = results['consensus_params'],
    param_defs       = PARAM_DEFS,
    fixed_params     = FIXED_PARAMS,
    threshold        = 0.20,
    show             = False,
    save_html        = None,
)

---
## Results Charts and Cost Stress Test

| Parameter | Description | Default |
|---|---|---|
| `show_fold_perf` | IS vs OOS bars for return, Sharpe, drawdown per fold | `False` |
| `show_param_evol` | Parameter evolution across folds with ±1 std bands | `False` |
| `show_oos_equity` | Combined OOS equity curve + drawdown with fold boundaries | `True` |
| `show_trades` | Overlay entry/exit markers on OOS equity chart | `False` |
| `benchmark_data` | DataFrame with `Close` column for buy & hold comparison | `None` |
| `save_html_dir` | Directory path to save charts as HTML files, or `None` | `None` |

**Cost Stress Test:** re-run the combined OOS backtest at 1×, 1.5×, 2×, 3× the base cost. Fragile strategies collapse; robust ones degrade gradually.

In [13]:
plot_walk_forward_results(
    results          = results,
    param_defs       = PARAM_DEFS,
    fixed_params     = FIXED_PARAMS,
    benchmark_data   = df,
    show             = True,
    save_html_dir    = None,
    show_fold_perf   = False,   # IS vs OOS bars by fold
    show_param_evol  = False,   # parameter evolution across folds
    show_oos_equity  = True,   # combined OOS equity curve
    show_trades      = False,  # trade markers on OOS equity chart
)

# ── transaction cost stress test ──────────────────────────────────────────

if results['oos_combined_df'] is not None:
    cost_df = cost_stress_test(
        oos_combined_df  = results['oos_combined_df'],
        cost_multipliers = (1.0, 1.5, 2.0, 3.0),
        base_cost        = COST,
    )
else:
    print('No combined OOS dataframe — skip cost stress test')


═══════════════════════════════════════════════════════════════════════════
TRANSACTION COST STRESS TEST
═══════════════════════════════════════════════════════════════════════════
    Cost   Mult   Sharpe     Return      MaxDD   Calmar       PF
──────── ────── ──────── ────────── ────────── ──────── ────────
  0.0010   1.0x     0.06     -5.34%    -30.27%    -0.06     1.21
  0.0015   1.5x    -0.03    -12.27%    -30.70%    -0.14     1.21
  0.0020   2.0x    -0.13    -18.69%    -31.34%    -0.21     1.21
  0.0030   3.0x    -0.32    -30.17%    -38.04%    -0.30     1.21
